In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/emineyetm/telco-customer-churn/__results__.html
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__resultx__.html
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__notebook__.ipynb
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__output__.json
/kaggle/input/notebooks/emineyetm/telco-customer-churn/churn_histogram.png
/kaggle/input/notebooks/emineyetm/telco-customer-churn/custom.css
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__results___files/__results___38_1.png
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__results___files/__results___24_14.png
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__results___files/__results___24_3.png
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__results___files/__results___20_0.png
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__results___files/__results___24_5.png
/kaggle/input/notebooks/emineyetm/telco-customer-churn/__results___files/__results___24_0.png
/kagg

In [2]:
import pandas as pd
import json
import os
from google.cloud import bigquery
from google.oauth2 import service_account
from kaggle_secrets import UserSecretsClient

# Load credentials from Kaggle Secrets
secrets = UserSecretsClient()
service_account_json = secrets.get_secret("GCP_SERVICE_ACCOUNT")
service_account_info = json.loads(service_account_json)

credentials = service_account.Credentials.from_service_account_info(
    service_account_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

PROJECT_ID = "churn-mlops"  # your GCP project ID
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

print("BigQuery client created successfully")

BigQuery client created successfully


In [3]:
# Load Telco dataset
df = pd.read_csv(
    '/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nChurn distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn rate: {(df['Churn']=='Yes').mean():.1%}")

Shape: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn rate: 26.5%


In [4]:
# Fix TotalCharges — has blank strings where tenure = 0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# Standardize column names to snake_case for BigQuery
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

print("Cleaned column names:")
print(list(df.columns))
print(f"\nNull check: {df.isnull().sum().sum()} total nulls")

Cleaned column names:
['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']

Null check: 0 total nulls


In [5]:
# Pre-upload data contract assertions
# Why: stops execution if data doesn't meet expectations
# so we never upload bad data to BigQuery

assert df.shape == (7043, 21), \
    f"Shape mismatch: {df.shape} — expected (7043, 21)"

assert df['customerid'].nunique() == 7043, \
    f"Duplicate customer IDs found: {7043 - df['customerid'].nunique()} duplicates"

assert set(df['churn'].unique()) == {'Yes', 'No'}, \
    f"Unexpected churn values: {df['churn'].unique()}"

assert df['monthlycharges'].min() >= 0, \
    "Negative monthly charges found — data quality issue"

assert df['tenure'].min() >= 0, \
    "Negative tenure found — data quality issue"

print("All pre-upload assertions passed")
print(f"  Rows:              {df.shape[0]:,}")
print(f"  Columns:           {df.shape[1]}")
print(f"  Unique customers:  {df['customerid'].nunique():,}")
print(f"  Churn rate:        {(df['churn']=='Yes').mean():.1%}")
print(f"  Tenure range:      {df['tenure'].min()}–{df['tenure'].max()} months")
print(f"  Charges range:     ${df['monthlycharges'].min()}–${df['monthlycharges'].max()}")
print("\nSafe to upload to BigQuery")

All pre-upload assertions passed
  Rows:              7,043
  Columns:           21
  Unique customers:  7,043
  Churn rate:        26.5%
  Tenure range:      0–72 months
  Charges range:     $18.25–$118.75

Safe to upload to BigQuery


In [6]:
# Define destination table
# Why churn_raw dataset: this is untransformed data — 
# the single source of truth before dbt touches it
table_id = f"{PROJECT_ID}.churn_raw.telco_customers"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True
)

print(f"Uploading {len(df):,} rows to {table_id}...")

job = client.load_table_from_dataframe(
    df, table_id, job_config=job_config
)
job.result()  # blocks until job completes

# Verify upload succeeded
table = client.get_table(table_id)
print(f"\nUpload successful")
print(f"  Destination:  {table_id}")
print(f"  Rows loaded:  {table.num_rows:,}")
print(f"  Columns:      {len(table.schema)}")
print(f"  Schema:")
for field in table.schema:
    print(f"    {field.name:<25} {field.field_type}")

Uploading 7,043 rows to churn-mlops.churn_raw.telco_customers...

Upload successful
  Destination:  churn-mlops.churn_raw.telco_customers
  Rows loaded:  7,043
  Columns:      21
  Schema:
    customerid                STRING
    gender                    STRING
    seniorcitizen             INTEGER
    partner                   STRING
    dependents                STRING
    tenure                    INTEGER
    phoneservice              STRING
    multiplelines             STRING
    internetservice           STRING
    onlinesecurity            STRING
    onlinebackup              STRING
    deviceprotection          STRING
    techsupport               STRING
    streamingtv               STRING
    streamingmovies           STRING
    contract                  STRING
    paperlessbilling          STRING
    paymentmethod             STRING
    monthlycharges            FLOAT
    totalcharges              FLOAT
    churn                     STRING


In [7]:
query = """
    SELECT
        COUNT(*)                                        AS total_accounts,
        SUM(CASE WHEN churn = 'Yes' THEN 1 END)        AS churned_accounts,
        ROUND(
            AVG(CASE WHEN churn = 'Yes' 
                THEN 1.0 ELSE 0 END) * 100, 1
        )                                               AS churn_rate_pct,
        ROUND(AVG(monthlycharges), 2)                   AS avg_monthly_charges,
        ROUND(MIN(monthlycharges), 2)                   AS min_monthly_charges,
        ROUND(MAX(monthlycharges), 2)                   AS max_monthly_charges,
        ROUND(AVG(tenure), 1)                           AS avg_tenure_months,
        COUNT(DISTINCT contract)                        AS distinct_contract_types,
        COUNT(DISTINCT paymentmethod)                   AS distinct_payment_methods
    FROM `churn-mlops.churn_raw.telco_customers`
"""

result = client.query(query).to_dataframe()

print("BigQuery verification query results:")
print(result.to_string(index=False))

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


BigQuery verification query results:
 total_accounts  churned_accounts  churn_rate_pct  avg_monthly_charges  min_monthly_charges  max_monthly_charges  avg_tenure_months  distinct_contract_types  distinct_payment_methods
           7043              1869            26.5                64.76                18.25               118.75               32.4                        3                         4
